**Random-walk policy**

The sensor repeatedly moves toward a uniformly sampled goal.

In [ ]:
!wget -q -O box_gym.py https://raw.githubusercontent.com/MurpheyLab/boxgpt/main/box_gym.py

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation
from tqdm.auto import tqdm

from box_gym import BoxGym


def show_video(frames):
    height, width = frames[0].shape[:2]
    dpi = 100
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax.axis("off")
    image = ax.imshow(frames[0])

    def update(index):
        image.set_data(frames[index])
        return (image,)

    video = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True
    )
    plt.close(fig)
    with plt.rc_context({"animation.embed_limit": 100.0}):
        return HTML(video.to_html5_video())

In [ ]:
def uniform_goal_action(observation, goal, env):
    direction = goal - observation["sensor_pos"]
    distance = np.linalg.norm(direction)
    if distance < 1e-12:
        return np.zeros(2, dtype=np.float32)
    return (direction / distance * env.max_velocity).astype(np.float32)

In [ ]:
env = BoxGym()
observation, info = env.reset(seed=12)
rng = np.random.default_rng(7)
diagnostics = True
max_steps = 300
goal_tolerance = 0.025
margin = env.sensor_size / 2
goal = rng.uniform(margin, 1.0 - margin, size=2)
frames = [env.render(diagnostics=diagnostics)]
goals_visited = 0

pbar = tqdm(range(max_steps))
for _ in pbar:
    if np.linalg.norm(observation["sensor_pos"] - goal) < goal_tolerance:
        goal = rng.uniform(margin, 1.0 - margin, size=2)
        goals_visited += 1

    action = uniform_goal_action(observation, goal, env)
    observation, reward, done, truncated, info = env.step(action)
    frames.append(env.render(diagnostics=diagnostics))
    pbar.set_description(f"uncertainty: {info['uncertainty']:.0e}")

    if done:
        break

env.close()
show_video(frames)